## Development

- databricks_simulated_retail_customer_data
- db_incoming_stream


### Retail-pipeline (customers)

##### readStream

In [0]:
# Import necessary functions
from pyspark.sql.functions import input_file_name, current_timestamp

# Configuration
source_path = "/Volumes/end_to_end_retail/db_landing/autoloader/incoming/customer_auto/"  #only one for now
checkpoint_path = "/Volumes/end_to_end_retail/db_landing/autoloader/incoming/customer_auto/_checkpoint_stream"
schema_location = "/Volumes/end_to_end_retail/db_landing/autoloader/incoming/customer_auto/"
target_table = "end_to_end_retail.db_landing.customer_autoloader"

#### Simple example for one file only
- add other later

In [0]:
customers_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_location)
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaHints", "timestamp DATE")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns") # Automatic drift handling
    .load(source_path)
    
    # .option("cloudFiles.useNotifications", "true")
    # .useNotifications will read from queues message coming from cloud services
    # AWS Event Notification, Azure Event Grid
    # .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
)

In [0]:
from pyspark.sql.functions import col, current_timestamp

customers_transformed_df = (
    customers_df.withColumn('file_path', col("_metadata.file_path"))
                .withColumn('ingestion_date', current_timestamp())
)

In [0]:
# 2. Write to Delta Table
query = (customers_transformed_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")                  # Allow target table to evolve
    .trigger(availableNow=True)                    # Process all new data and stop
    .toTable(target_table)
)

query.awaitTermination()

In [0]:
query.stop()

#### readStream tracking columns

In [0]:
%sql
SELECT * FROM end_to_end_retail.db_landing.customer_autoloader limit 5

#### retail-pipeline (orders)

In [0]:
# Import necessary functions
from pyspark.sql.functions import input_file_name, current_timestamp

# Configuration
source_path = "/Volumes/end_to_end_retail/db_landing/autoloader/incoming/retail_pipeline_orders/"  #only one for now
checkpoint_path = "/Volumes/end_to_end_retail/db_landing/autoloader/incoming/retail_pipeline_orders/_checkpoint_stream"
schema_location = "/Volumes/end_to_end_retail/db_landing/autoloader/incoming/retail_pipeline_orders/"
target_table = "end_to_end_retail.db_landing.retail_pipeline_autoloader"

In [0]:
retail_orders_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_location)
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaHints", "order_timestamp DATE")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns") # Automatic drift handling
    .load(source_path)
    
    # .option("cloudFiles.useNotifications", "true")
    # .useNotifications will read from queues message coming from cloud services
    # AWS Event Notification, Azure Event Grid
    # .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
)

In [0]:
from pyspark.sql.functions import col, current_timestamp

retail_orders_transformed_df = (
    retail_orders_df.withColumn('file_path', col("_metadata.file_path"))
                .withColumn('ingestion_date', current_timestamp())
)

In [0]:
# 2. Write to Delta Table
query = (retail_orders_transformed_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")                  # Allow target table to evolve
    .trigger(availableNow=True)                    # Process all new data and stop
    .toTable(target_table)
)

query.awaitTermination()

In [0]:
query.stop()

In [0]:
%sql
SELECT * FROM end_to_end_retail.db_landing.retail_pipeline_autoloader limit 5

In [0]:
# Load JSON file
df = spark.read.json("/Volumes/databricks_simulated_retail_customer_data/v01/retail-pipeline/orders/stream_json/00.json")
display(df.head(5))

#### retail-pipeline (status)

In [0]:
# Import necessary functions
from pyspark.sql.functions import input_file_name, current_timestamp

# Configuration
source_path = "/Volumes/end_to_end_retail/db_landing/autoloader/incoming/retail_pipeline_status/"  #only one for now
checkpoint_path = "/Volumes/end_to_end_retail/db_landing/autoloader/incoming/retail_pipeline_status/_checkpoint_stream"
schema_location = "/Volumes/end_to_end_retail/db_landing/autoloader/incoming/retail_pipeline_status/"
target_table = "end_to_end_retail.db_landing.retail_pipeline_status_autoloader"

In [0]:
retail_status_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_location)
    .option("cloudFiles.inferColumnTypes", "true")
    # .option("cloudFiles.schemaHints", "status_timestamp DATE")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns") # Automatic drift handling
    .load(source_path)
    
    # .option("cloudFiles.useNotifications", "true")
    # .useNotifications will read from queues message coming from cloud services
    # AWS Event Notification, Azure Event Grid
    # .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
)

In [0]:
from pyspark.sql.functions import col, current_timestamp

retail_status_transformed_df = (
    retail_status_df.withColumn('file_path', col("_metadata.file_path"))
                .withColumn('ingestion_date', current_timestamp())
)

In [0]:
# 2. Write to Delta Table
query = (retail_status_transformed_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")                  # Allow target table to evolve
    .trigger(availableNow=True)                    # Process all new data and stop
    .toTable(target_table)
)

query.awaitTermination()

In [0]:
query.stop()

In [0]:
%sql
SELECT * FROM end_to_end_retail.db_landing.retail_pipeline_status_autoloader limit 5


In [0]:
# Load JSON file
df = spark.read.json("/Volumes/databricks_simulated_retail_customer_data/v01/retail-pipeline/status/stream_json/00.json")
display(df.head(5))